# Create New dataset with 0, 1, 2, 3, 4, 5 examples.


In [ ]:
import json
import pandas as pd

SOURCE = "data/solver_results_forcing_base_10_k_1.parquet"

src = pd.read_parquet(SOURCE)
print(f"Source: {len(src)} rows")

In [ ]:
# --- Shared helpers (reused by the n=1..5 cells below) ---

INTRO = (
    "In Alice's Wonderland, a secret set of transformation rules is applied to equations. "
    "Below are a few examples:"
)

def lhs_str(l: dict) -> str:
    return l["l0"] + l["l1"] + l["op"] + l["r0"] + l["r1"]

def rhs_str(rh: dict) -> str:
    body = "".join(rh["actual_rhs"])
    if rh["op"] is None:
        return body
    return rh["op"] + body if rh["sign_pos"] == "prefix" else body + rh["op"]

def make_prompt(example_lines: list[str], target_lhs: str) -> str:
    """Build the Alice prompt.  example_lines is a list of 'lhs = rhs' strings."""
    body = "\n".join(example_lines)
    sep = "\n" if example_lines else ""
    return f"{INTRO}\n{body}{sep}\nNow, determine the result for: {target_lhs}"

## n = 0 examples
Each equation in every source row becomes one zero-shot sample (no worked examples shown).

In [ ]:
rows_0 = []
for _, r in src.iterrows():
    for i, eq in enumerate(json.loads(r["equations"])):
        rows_0.append({
            "id":     f"{r['id']}_{i}",
            "prompt": make_prompt([], lhs_str(eq["lhs"])),
            "answer": rhs_str(eq["rhs"]),
        })

df0 = pd.DataFrame(rows_0)
print(f"Rows: {len(df0)}")
df0.head()

In [ ]:
print(df0['prompt'].iloc[0])

In [ ]:
# Sanity checks
assert len(df0) == 4165,       f"Expected 4165 rows, got {len(df0)}"
assert df0["id"].nunique() == 4165, "IDs are not unique"
assert df0.isnull().sum().sum() == 0, "Unexpected nulls"

# Verify to_predict answers match source
for _, r in src.iterrows():
    eqs = json.loads(r["equations"])
    tp_idx = next(i for i, e in enumerate(eqs) if e["type"] == "to_predict")
    derived = df0.loc[df0["id"] == f"{r['id']}_{tp_idx}", "answer"].iloc[0]
    assert derived == r["answer"], f"Answer mismatch for {r['id']}"

print("All checks passed ✓")

# Spot-check a sample
sample = df0.sample(1, random_state=42).iloc[0]
print(f"\n--- sample id: {sample['id']} ---")
print(sample["prompt"])
print("answer:", sample["answer"])

In [ ]:
OUTPUT_0 = "data/dataset_examples_0.parquet"
df0.to_parquet(OUTPUT_0, index=False)
print(f"Saved → {OUTPUT_0}  shape={df0.shape}")

## n = 1..5 examples — leave-one-out query rotation

**Design:** every equation in a row takes one turn as the query; n examples are
chosen from the remaining equations. One sample per `(row, query)` unit.

**Source:** all 823 rows — arithmetic and `pure_concat` treated uniformly.
The solver decides downstream which exercises fall under `pure_concat`; a
rotated concat equation can produce a *covered* or *uncovered* sample that the
solver handles differently. No branching is done here.

**Coverage rule:** target **80.07 % covered / 19.93 % uncovered** (= 659/164,
the coverage of the full 823-row source measured on the original `to_predict`
equations).  Coverage is held at this ratio by assigning the *flexible* units.
At n=5 there are no flexible units — all units are either `forced_cov` or
`forced_unc` — so coverage drifts to the natural value (~86.77 %) and all 1716
eligible units are included.

**All eligible units are kept** (no down-selection). Each `(row, query)` unit
with `≥ n` other equations in the pool becomes exactly one sample.

Unit classification by pool (other equations in the row):
- `forced_cov` — coverable only  (≥1 matching op, but <n non-matching ops)
- `forced_unc` — uncoverable only (0 matching ops)
- `flexible`   — both (≥1 matching AND ≥n non-matching; assigned by RNG)

Achieved sizes:

| n | total | covered | uncovered | C% |
|---|---|---|---|---|
| 1 | 4165 | 3335 | 830 | 80.07% |
| 2 | 4165 | 3335 | 830 | 80.07% |
| 3 | 4165 | 3335 | 830 | 80.07% |
| 4 | 3221 | 2579 | 642 | 80.07% |
| 5 | 1716 | 1489 | 227 | 86.77% (natural drift) |

n=1..3 equal the n=0 dataset size (4165) — the whole n=0..5 family spans the
same 823-row source.

In [ ]:
import random

SOURCE_ALL = "data/solver_results_forcing_base_10_k_1.parquet"
SEED       = 42
COV_FRAC   = 659 / 823   # 80.07 % — coverage of the full 823-row source

src_all = pd.read_parquet(SOURCE_ALL)
print(f"Source: {len(src_all)} rows")


def build_nshot(n: int, seed: int = SEED) -> pd.DataFrame:
    """Leave-one-out rotation dataset with n shown examples.

    All 823 rows are treated as one uniform pool — no concat / non-concat split.
    The solver decides downstream what is pure_concat.

    Unit classification by pool (all other equations in the row):
      forced_cov  — ≥1 matching op, <n non-matching  → must be covered
      forced_unc  — 0 matching ops                    → must be uncovered
      flexible    — ≥1 matching AND ≥n non-matching   → assigned to hit target

    Target coverage: COV_FRAC (659/823 ≈ 80.07 %).
    All eligible units are kept (no down-selection).
    If forced units already exceed the target for either class, the flexible
    units all go to the other class and coverage drifts naturally (n=5 case).
    """
    rng = random.Random(seed)

    # ── Phase 1: enumerate and classify all eligible units ────────────────────
    forced_cov: list = []
    forced_unc: list = []
    flexible:   list = []

    for _, row in src_all.iterrows():
        eqs = json.loads(row["equations"])
        for qi, q in enumerate(eqs):
            pool_idx    = [j for j in range(len(eqs)) if j != qi]
            if len(pool_idx) < n:
                continue
            qop         = q["lhs"]["op"]
            match_idx   = [j for j in pool_idx if eqs[j]["lhs"]["op"] == qop]
            nomatch_idx = [j for j in pool_idx if eqs[j]["lhs"]["op"] != qop]
            unit = (row["problem_id"], qi, eqs, q, pool_idx, match_idx, nomatch_idx)
            if match_idx and len(nomatch_idx) < n:
                forced_cov.append(unit)
            elif not match_idx:
                forced_unc.append(unit)
            else:
                flexible.append(unit)

    # ── Phase 2: assign flexible units to hit COV_FRAC (keep all units) ───────
    all_units = len(forced_cov) + len(forced_unc) + len(flexible)
    tC = round(all_units * COV_FRAC)
    tU = all_units - tC

    need_cov_flex = tC - len(forced_cov)
    need_unc_flex = tU - len(forced_unc)

    if need_cov_flex < 0:
        # forced_cov already exceeds target → all flexible go to uncovered
        cov_flex, unc_flex = 0, len(flexible)
    elif need_unc_flex < 0:
        # forced_unc already exceeds target → all flexible go to covered
        cov_flex, unc_flex = len(flexible), 0
    else:
        cov_flex, unc_flex = need_cov_flex, need_unc_flex

    rng.shuffle(flexible)
    chosen_cov = forced_cov + flexible[:cov_flex]
    chosen_unc = forced_unc + flexible[cov_flex: cov_flex + unc_flex]

    C, U = len(chosen_cov), len(chosen_unc)

    # ── Phase 3: build samples ────────────────────────────────────────────────
    rows_out = []
    for label, units in [("cov", chosen_cov), ("unc", chosen_unc)]:
        for row_id, qi, eqs, q, pool_idx, match_idx, nomatch_idx in units:
            if label == "cov":
                anchor = rng.sample(match_idx, 1)
                rest   = rng.sample([j for j in pool_idx if j not in anchor], n - 1)
                ex_idx = anchor + rest
            else:
                ex_idx = rng.sample(nomatch_idx, n)
            rng.shuffle(ex_idx)
            ex_lines = [
                f"{lhs_str(eqs[j]['lhs'])} = {rhs_str(eqs[j]['rhs'])}"
                for j in ex_idx
            ]
            rows_out.append({
                "id":     f"{row_id}_q{qi}",
                "prompt": make_prompt(ex_lines, lhs_str(q["lhs"])),
                "answer": rhs_str(q["rhs"]),
            })

    df = pd.DataFrame(rows_out)
    print(f"n={n}: {len(df)} rows  C={C} ({C/len(df)*100:.2f}%)  U={U} ({U/len(df)*100:.2f}%)")
    return df

In [ ]:
EXPECTED   = {1: 4165, 2: 4165, 3: 4165, 4: 3221, 5: 1716}
TARGET_COV = {1: 0.8007, 2: 0.8007, 3: 0.8007, 4: 0.8007, 5: 0.8677}  # natural drift at n=5
TOL        = 0.005   # ±0.5 %

for n in range(1, 6):
    df = build_nshot(n)

    # ── Structural checks ────────────────────────────────────────────────────
    assert len(df) == EXPECTED[n],        f"n={n}: expected {EXPECTED[n]} rows, got {len(df)}"
    assert df["problem_id"].nunique() == len(df), f"n={n}: duplicate ids"
    assert df.isnull().sum().sum() == 0,  f"n={n}: nulls found"

    # Each prompt must have exactly n example lines (lines containing " = ")
    bad_ex = df["prompt"].apply(
        lambda p: sum(1 for ln in p.split("\n") if " = " in ln) != n
    ).sum()
    assert bad_ex == 0, f"n={n}: {bad_ex} prompts have wrong number of example lines"

    # Last line (query) must not contain " = "
    bad_tail = df["prompt"].apply(lambda p: " = " in p.split("\n")[-1]).sum()
    assert bad_tail == 0, f"n={n}: {bad_tail} prompts have '=' in the query line"

    # Coverage fraction check (all rows; op glyph at lhs position 2)
    def is_covered(prompt: str) -> bool:
        lines     = prompt.split("\n")
        query_lhs = lines[-1].split(": ", 1)[1]
        query_op  = query_lhs[2]
        return any(ln.split(" = ")[0][2] == query_op for ln in lines if " = " in ln)

    C_frac = df["prompt"].apply(is_covered).mean()
    assert abs(C_frac - TARGET_COV[n]) < TOL, \
        f"n={n}: coverage {C_frac:.4f} not within ±{TOL} of {TARGET_COV[n]:.4f}"

    print(f"  n={n} ✓  coverage={C_frac*100:.2f}%")

    # Spot-check one sample
    s = df.sample(1, random_state=n).iloc[0]
    print(f"  sample {s['id']}:  {s['prompt'].split(chr(10))[-1]}  →  {s['answer']}")

    # Save
    out = f"data/dataset_examples_{n}.parquet"
    df.to_parquet(out, index=False)
    print(f"  Saved → {out}\n")